# Roof Generation with topologic_fast

This notebook demonstrates roof generation and operations using topologic_fast.
We will create various building footprints and generate roof geometries.

**Note**: This is adapted from the topologicpy Roof example.

**Note**: This notebook uses topologic_fast's native `Cell.Roof()` implementation which supports
gable, hip, flat, and shed roof types.

In [ ]:
import topologic_fast as tf
import plotly.graph_objects as go
import math

## 1. Create Building Base

First, let's create a simple building that we can add a roof to.

In [ ]:
# Create a building box (20m x 10m x 6m)
building_width = 20
building_length = 10
building_height = 6

building = tf.Cell.Box(0, 0, 0, building_width, building_length, building_height)

print(f"Building dimensions: {building_width}m x {building_length}m x {building_height}m")
print(f"Building volume: {building.Volume():.2f} m^3")

## 2. Visualization Helper Functions

In [ ]:
def cell_to_mesh_data(cell, color='lightblue', opacity=0.7, name='Cell'):
    """
    Convert a Cell to Plotly Mesh3d trace data.
    """
    faces = cell.Faces()
    
    all_x, all_y, all_z = [], [], []
    all_i, all_j, all_k = [], [], []
    vertex_idx = 0
    
    for face in faces:
        vertices = face.Vertices()
        if len(vertices) >= 3:
            coords = [v.Coordinates() for v in vertices]
            
            base_idx = vertex_idx
            for coord in coords:
                all_x.append(coord[0])
                all_y.append(coord[1])
                all_z.append(coord[2])
                vertex_idx += 1
            
            for i in range(1, len(coords) - 1):
                all_i.append(base_idx)
                all_j.append(base_idx + i)
                all_k.append(base_idx + i + 1)
    
    return go.Mesh3d(
        x=all_x, y=all_y, z=all_z,
        i=all_i, j=all_j, k=all_k,
        color=color,
        opacity=opacity,
        name=name,
        flatshading=True
    )

def face_to_mesh_data(face, color='red', opacity=0.8, name='Face'):
    """
    Convert a Face to Plotly Mesh3d trace data.
    """
    vertices = face.Vertices()
    coords = [v.Coordinates() for v in vertices]
    
    x = [c[0] for c in coords]
    y = [c[1] for c in coords]
    z = [c[2] for c in coords]
    
    i, j, k = [], [], []
    for idx in range(1, len(coords) - 1):
        i.append(0)
        j.append(idx)
        k.append(idx + 1)
    
    return go.Mesh3d(
        x=x, y=y, z=z,
        i=i, j=j, k=k,
        color=color,
        opacity=opacity,
        name=name,
        flatshading=True
    )

def faces_to_mesh_data(faces, color='red', opacity=0.8, name='Faces'):
    """
    Convert multiple Faces to a single Plotly Mesh3d trace.
    """
    all_x, all_y, all_z = [], [], []
    all_i, all_j, all_k = [], [], []
    vertex_idx = 0
    
    for face in faces:
        vertices = face.Vertices()
        if len(vertices) >= 3:
            coords = [v.Coordinates() for v in vertices]
            
            base_idx = vertex_idx
            for coord in coords:
                all_x.append(coord[0])
                all_y.append(coord[1])
                all_z.append(coord[2])
                vertex_idx += 1
            
            for i in range(1, len(coords) - 1):
                all_i.append(base_idx)
                all_j.append(base_idx + i)
                all_k.append(base_idx + i + 1)
    
    return go.Mesh3d(
        x=all_x, y=all_y, z=all_z,
        i=all_i, j=all_j, k=all_k,
        color=color,
        opacity=opacity,
        name=name,
        flatshading=True
    )

def cell_to_edges(cell, color='black', width=1):
    """
    Convert a Cell's edges to Plotly Scatter3d trace.
    """
    edges = cell.Edges()
    x_edges, y_edges, z_edges = [], [], []
    
    for edge in edges:
        verts = edge.Vertices()
        p1 = verts[0].Coordinates()
        p2 = verts[1].Coordinates()
        x_edges.extend([p1[0], p2[0], None])
        y_edges.extend([p1[1], p2[1], None])
        z_edges.extend([p1[2], p2[2], None])
    
    return go.Scatter3d(
        x=x_edges, y=y_edges, z=z_edges,
        mode='lines',
        line=dict(color=color, width=width),
        showlegend=False
    )

## 3. Create Roof Types Using topologic_fast

topologic_fast provides `Cell.Roof()` to create various roof types directly.

In [ ]:
# Create gable roof using topologic_fast's native implementation
# Cell.Roof creates a roof cell from a base face
# roof_type can be "gable", "hip", "flat", "shed"

# Get the top face of the building as the base
top_face = None
for face in building.Faces():
    center = face.CenterOfMass()
    if abs(center[2] - building_height) < 0.1:
        # Check if this is a horizontal face (top or bottom)
        normal = face.Normal()
        if abs(normal[2]) > 0.9:  # Z-component close to 1
            top_face = face
            break

# Create gable roof
gable_roof = tf.Cell.Roof(top_face, roof_type="gable", angle=30)
roof_faces = gable_roof.Faces()

print(f"Created gable roof with {len(roof_faces)} faces")
print(f"Roof volume: {gable_roof.Volume():.2f} m^3")
total_area = sum(f.Area() for f in roof_faces)
print(f"Total roof surface area: {total_area:.2f} m^2")

## 4. Visualize Building with Gable Roof

In [ ]:
fig = go.Figure()

# Add building
fig.add_trace(cell_to_mesh_data(building, color='lightgray', opacity=0.7, name='Building'))
fig.add_trace(cell_to_edges(building, color='gray'))

# Add roof faces
fig.add_trace(faces_to_mesh_data(roof_faces, color='brown', opacity=0.9, name='Gable Roof'))

# Add roof edges
for face in roof_faces:
    edges = face.Edges()
    for edge in edges:
        verts = edge.Vertices()
        p1 = verts[0].Coordinates()
        p2 = verts[1].Coordinates()
        fig.add_trace(go.Scatter3d(
            x=[p1[0], p2[0]], y=[p1[1], p2[1]], z=[p1[2], p2[2]],
            mode='lines',
            line=dict(color='saddlebrown', width=3),
            showlegend=False
        ))

fig.update_layout(
    title='Building with Gable Roof',
    scene=dict(
        aspectmode='data',
        xaxis_title='X (m)',
        yaxis_title='Y (m)',
        zaxis_title='Z (m)',
        camera=dict(eye=dict(x=1.5, y=1.5, z=1.0))
    ),
    width=900,
    height=700
)

fig.show()

## 5. Create Hip Roof

A hip roof has four sloped sides that all come together at a ridge.

In [ ]:
# Create hip roof using topologic_fast's native implementation
hip_roof = tf.Cell.Roof(top_face, roof_type="hip", angle=30)
hip_roof_faces = hip_roof.Faces()

print(f"Created hip roof with {len(hip_roof_faces)} faces")
print(f"Roof volume: {hip_roof.Volume():.2f} m^3")
total_area = sum(f.Area() for f in hip_roof_faces)
print(f"Total roof surface area: {total_area:.2f} m^2")

## 6. Visualize Building with Hip Roof

In [ ]:
fig = go.Figure()

# Add building
fig.add_trace(cell_to_mesh_data(building, color='lightgray', opacity=0.7, name='Building'))
fig.add_trace(cell_to_edges(building, color='gray'))

# Add hip roof faces
fig.add_trace(faces_to_mesh_data(hip_roof_faces, color='firebrick', opacity=0.9, name='Hip Roof'))

# Add roof edges
for face in hip_roof_faces:
    edges = face.Edges()
    for edge in edges:
        verts = edge.Vertices()
        p1 = verts[0].Coordinates()
        p2 = verts[1].Coordinates()
        fig.add_trace(go.Scatter3d(
            x=[p1[0], p2[0]], y=[p1[1], p2[1]], z=[p1[2], p2[2]],
            mode='lines',
            line=dict(color='darkred', width=3),
            showlegend=False
        ))

fig.update_layout(
    title='Building with Hip Roof',
    scene=dict(
        aspectmode='data',
        xaxis_title='X (m)',
        yaxis_title='Y (m)',
        zaxis_title='Z (m)',
        camera=dict(eye=dict(x=1.5, y=1.5, z=1.0))
    ),
    width=900,
    height=700
)

fig.show()

## 7. Create Pyramid Roof

For a square building, a pyramid roof with four triangular faces meeting at a point.
This is achieved with a hip roof on a square footprint.

In [ ]:
# Create a square building
square_size = 12
square_building = tf.Cell.Box(0, 0, 0, square_size, square_size, 5)

# Get top face
top_face_square = None
for face in square_building.Faces():
    center = face.CenterOfMass()
    if abs(center[2] - 5) < 0.1:
        normal = face.Normal()
        if abs(normal[2]) > 0.9:
            top_face_square = face
            break

# Create pyramid roof (hip roof on square = pyramid)
pyramid_roof = tf.Cell.Roof(top_face_square, roof_type="hip", angle=45)
pyramid_roof_faces = pyramid_roof.Faces()

print(f"Created pyramid roof with {len(pyramid_roof_faces)} faces")
total_area = sum(f.Area() for f in pyramid_roof_faces)
print(f"Total roof area: {total_area:.2f} m^2")

## 8. Visualize Building with Pyramid Roof

In [ ]:
fig = go.Figure()

# Add building
fig.add_trace(cell_to_mesh_data(square_building, color='lightgray', opacity=0.7, name='Building'))
fig.add_trace(cell_to_edges(square_building, color='gray'))

# Add pyramid roof faces
fig.add_trace(faces_to_mesh_data(pyramid_roof_faces, color='darkgreen', opacity=0.9, name='Pyramid Roof'))

# Add roof edges
for face in pyramid_roof_faces:
    edges = face.Edges()
    for edge in edges:
        verts = edge.Vertices()
        p1 = verts[0].Coordinates()
        p2 = verts[1].Coordinates()
        fig.add_trace(go.Scatter3d(
            x=[p1[0], p2[0]], y=[p1[1], p2[1]], z=[p1[2], p2[2]],
            mode='lines',
            line=dict(color='darkgreen', width=3),
            showlegend=False
        ))

fig.update_layout(
    title='Building with Pyramid Roof',
    scene=dict(
        aspectmode='data',
        xaxis_title='X (m)',
        yaxis_title='Y (m)',
        zaxis_title='Z (m)',
        camera=dict(eye=dict(x=1.5, y=1.5, z=1.0))
    ),
    width=900,
    height=700
)

fig.show()

## 9. Create Shed Roof

A shed roof has a single sloped surface.

In [ ]:
# Create shed roof using topologic_fast's native implementation
# direction specifies which way the roof slopes (0=+Y, 90=+X, 180=-Y, 270=-X)
shed_roof = tf.Cell.Roof(top_face, roof_type="shed", angle=20, direction=0)
shed_roof_faces = shed_roof.Faces()

print(f"Created shed roof with {len(shed_roof_faces)} faces")
print(f"Roof volume: {shed_roof.Volume():.2f} m^3")

## 10. Compare All Roof Types

In [ ]:
from plotly.subplots import make_subplots

# Create separate buildings for comparison at different positions
buildings = [
    tf.Cell.Box(0, 0, 0, 10, 8, 4),    # Gable
    tf.Cell.Box(15, 0, 0, 10, 8, 4),   # Hip
    tf.Cell.Box(0, 15, 0, 8, 8, 4),    # Pyramid (square base)
    tf.Cell.Box(15, 15, 0, 10, 8, 4),  # Shed
]

# Get top faces for each building
def get_top_face(building, height):
    """Get the top face of a building."""
    for face in building.Faces():
        center = face.CenterOfMass()
        if abs(center[2] - height) < 0.1:
            normal = face.Normal()
            if abs(normal[2]) > 0.9:
                return face
    return None

top_faces = [
    get_top_face(buildings[0], 4),
    get_top_face(buildings[1], 4),
    get_top_face(buildings[2], 4),
    get_top_face(buildings[3], 4),
]

# Create roofs using tf.Cell.Roof
roofs = []
roof_names = ['Gable', 'Hip', 'Pyramid', 'Shed']
roof_colors = ['brown', 'firebrick', 'darkgreen', 'steelblue']

for i, (top_face, name) in enumerate(zip(top_faces, roof_names)):
    if top_face is not None:
        if name == 'Gable':
            roof = tf.Cell.Roof(top_face, roof_type="gable", angle=30)
        elif name == 'Hip':
            roof = tf.Cell.Roof(top_face, roof_type="hip", angle=30)
        elif name == 'Pyramid':
            roof = tf.Cell.Roof(top_face, roof_type="hip", angle=45)  # Hip on square = pyramid
        else:  # Shed
            roof = tf.Cell.Roof(top_face, roof_type="shed", angle=20, direction=0)
        roofs.append(roof.Faces())
    else:
        roofs.append([])

# Create figure
fig = go.Figure()

# Add all buildings and roofs
for i, (bldg, roof_faces, color, name) in enumerate(zip(buildings, roofs, roof_colors, roof_names)):
    fig.add_trace(cell_to_mesh_data(bldg, color='lightgray', opacity=0.6, name=f'{name} Building'))
    fig.add_trace(cell_to_edges(bldg, color='gray'))
    if roof_faces:
        fig.add_trace(faces_to_mesh_data(roof_faces, color=color, opacity=0.9, name=f'{name} Roof'))

# Add labels
label_positions = [(5, 4, 10), (20, 4, 10), (4, 19, 10), (20, 19, 10)]
for (x, y, z), name in zip(label_positions, roof_names):
    fig.add_trace(go.Scatter3d(
        x=[x], y=[y], z=[z],
        mode='text',
        text=[name],
        textposition='middle center',
        textfont=dict(size=14, color='black'),
        showlegend=False
    ))

fig.update_layout(
    title='Comparison of Roof Types',
    scene=dict(
        aspectmode='data',
        xaxis_title='X (m)',
        yaxis_title='Y (m)',
        zaxis_title='Z (m)',
        camera=dict(eye=dict(x=1.2, y=1.2, z=1.0))
    ),
    width=1000,
    height=800
)

fig.show()

## Summary

This notebook demonstrated creating various roof types using topologic_fast:

1. **Gable Roof** - Two sloped surfaces meeting at a ridge, with triangular gable ends
2. **Hip Roof** - Four sloped surfaces, all meeting at a ridge
3. **Pyramid Roof** - Four triangular surfaces meeting at a central apex (hip roof on square)
4. **Shed Roof** - Single sloped surface (mono-pitch)

### topologic_fast Methods Used:

- `Cell.Roof(face, roof_type, angle, direction)` - Creates a roof cell from a base face
  - `roof_type`: "gable", "hip", "flat", "shed"
  - `angle`: Roof pitch angle in degrees
  - `direction`: For shed roofs, direction of slope (0, 90, 180, 270 degrees)

- `Cell.Box()` - Create building geometry
- `cell.Faces()` - Get all faces of a cell
- `face.CenterOfMass()` - Get center point
- `face.Normal()` - Get face normal direction

### Applications:
- Architectural modeling
- Building energy analysis
- Solar panel placement optimization
- Rainwater collection design